## Agents

- Agent Steps:

User--> Model--> Tools-->  Call tools--> Observe the result--> Continue reasoning--> Final answer

  - Model understand the task, split the job, understand whether to call the tools 
  - Tools are the functions that specialize in one area 
  - Memory to remember the context, call tools result and the users perference. 
  - Loop:  Plan Act Observe 

Langchain  Agent: 
Call model --> Tool execution(iterate)

- Basic concepts of each parts

  - Tools:  
    1. LLM has no real time information
    2. Can't access to local private data 
    3. Can't do computation 
    4. Can't run the code or do outside execution 

    Given: messages, tool schemas, tool executor

    repeat:
        1. 把 messages + tool schemas 发给 LLM
        2. LLM 返回： final answer or tool call( name, args)
        3. 如果是 tool call:
            执行对应函数
            把结果作为 tool output 加回 messages ( add to the  prompt)
        4. 如果是 final answer:
            stop

  
    - Plan Act and Observe 
    1. Plan: LLM determined whether the question needs to call the tools, which tools, the argument in the tools. If looks up into the document then compute ...

    2. Act: Call the function tools 

    3. Observe: Give the result to the LLM. Check if the result is enough? if need to call the next step tools? if need to do the correction? if the question can be answered. 


    - Difference in Pipeline and the Agents flow
    Previous LLM application is more related to the pipeline. The pipeline is comprised of question, RAG, prompt into LLM and output the result. 
    Thus **fixed** steps. 

    Agents Flow: The model dynamics to decide the next steps. The LLM will determine if doing the act? observing the result? if calling other tools, and then summary. More **flexible** steps. 



    - Multistep reasoning.
    When processing a big complex problem, directly calling the LLM is hard to get the answers. The big complex problem needs to be split into the small task. 
    
    Exp: Help compare the three insurance company and then recommend. 
    Split into : 1. Get the schema information
                 2. Alignment 
                 3. Compute the price and the insurance content. 
                 4. Summary and give the reasons. 

      - Steps for the multiple reasoning. 
      Plan act observe then update and summary 
      1. Split jobs
      2. Determine the orders 
      3. Call the functions if needy. 
      4. Remember the result, which steps current is in. 
      5. Self-check, if the result is contradictional? If need to call the function again? 
      6. Summary 



      - Memory

        - Short term memory: 当前会话的记忆。当前message plan，result， the loaded document。通过维护一个message 列表可实现。 每次append **（role， name， content）**。 checkpoint helps trans into longterm memory 

        - Long term memory: Cross sessions storage。 比如用户的身份， 偏好， 历史项目， 还有长期的事实。 
        通过维护一个json file 。 User_id is the key for the long term memory。通过写入key 对指定的user id 可以更新他的内容。 

        Example LONG_TERM_MEMORY = {
            "user_123": {
                "language": "Chinese",
                "research_interest": "scientific machine learning",
                "prefers_examples": True
            }
        }
      
        LangGraph / LangChain 官方把 memory 分得很清楚：：state 是短期记忆，store 是长期记忆 
  
 



# A small custom agent should have the following functions
To check the weather 
1. planner : decide the action 
2. tool executor: Tools
3. observation memory: it may have the long term memory or the short term memory 
4. final synethesis  

In [ ]:
import re
from typing import Dict, Any, List


question= 'check the weahter of beijing and shanghai, what is the difference' \n
'and what is the answer of 30 * 10'

# The angent should be able to determine if calling the weather agent, the calculator 
# Loop until it outputs the final answer. 


In [ ]:
# two tools for calling 

# calculator
def calculator(expression: str) -> str:
    allowed = "0123456789+-*/(). "
    if any(ch not in allowed for ch in expression):
        return "Error: invalid characters"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"

# weather
def get_weather(city: str, day: str = "tomorrow") -> str:
    # mock data
    fake_weather = {
        ("北京", "tomorrow"): 20,
        ("上海", "tomorrow"): 25,
    }
    temp = fake_weather.get((city, day), None)
    if temp is None:
        return f"No weather data for {city}"
    return f"{city} {day} temperature is {temp} C"

# organize the tool function into a tool dict 
Tools = { 
         "get_weather": get_weather, 
         "calculator": calculator 
         }


In [ ]:
# Planer 

def decide_action( user_query: str) -> List[ Dict[str, Any]]: 
    # return a list of dictionary 
    
    actions = [] 
    # a list of dictionary. 
    # The dict has the tools the args as the key , args have the arguments as the key 
    if "weather" in user_query or "temperature" in user_query: 
        if "beijing" in user_query: 
            actions.append({"tools": "get_weather", "args": 
                {"city":"beijing", "day": "tomorrow"}}) 
            
        if "shanghai" in user_query: 
            actions.append({"tools": "get_weather", "args": 
                {"city":"shanghai", "day": "tomorrow"}}) 
            
    
    # from the user query find  if there is any computation need to call the calculator
    expr_match = re.findall(r"[0-9+\-*/(). ]+", user_query)
    expr_match = [x.strip() for x in expr_match if x.strip() and any(ch.isdigit() for ch in x)]
    
    # find needs to call the calculator tools
    if  expr_match: 
        expression = max(expr_match, key=len)
        
        actions.append({"tools": "calculator", 'args':{"expression": expression}})
        
    return actions 
          
                               

In [ ]:
# Agent Loop 

def custom_agent(user_query: str ) -> str: 
    # store the current query and the action returned result,
    # these two as the keys 
    short_term_memory = { 
         "user_query": user_query, 
         "tools_results": []            
                    }

    # returns a list of the action with the arguments 
    actions = detect_action(user_query)
    
    
    # Loop the action , do the Act , to get the result 
    # The result store in the short term memory ,stroing with the dict
    # Where the key is the tools_result with the tool name , args and the returned result 
    for action in actions: 
        tool_name = action['tools']
        args = action['args'] # call can **args 
        
        result = Tools[tool_name](**args) 
        
        short_term_memory['tools_results'].append(
            {"tools":tool_name,
             "args": args,
             "result": result  
             }
           ) 
        
        
    # Final synthesis 
    weather_info = []
    calc_result = None 
    
    # extract the results from the stored short memory 
    
    
    for item in short_term_memory["tools_results"]:
        if item['tools'] == 'get_weather': 
            weather_info.append(item['result'])
            
        elif item['tools'] == 'caculator': 
            calc_result.append( item['result']) 
            
    #organize 
    final_parts = [] 
    
    if weather_info: 
        final_parts.extend(weather_info)
        
        
        temps = []
        for text in weather_info:
            
            m = re.search(r"is (\d+) C", text)
            if m:
                temps.append(int(m.group(1)))
        if len(temps) >= 2:
            final_parts.append(f"Temperature difference is {abs(temps[0] - temps[1])} C.")

        
        
    if calc_result is not None:
        final_parts.append(f"25*17 = {calc_result}")  
    

    return "\n".join(final_parts)
    
    


In [ ]:
import re
from typing import Dict, Any, List


# make a more general cases
# ----------------------
# Mock weather database
# ----------------------
FAKE_WEATHER = {
    "北京": {"today": 18, "tomorrow": 20},
    "上海": {"today": 22, "tomorrow": 25},
    "广州": {"today": 26, "tomorrow": 28},
    "深圳": {"today": 27, "tomorrow": 29},
    "杭州": {"today": 21, "tomorrow": 24},
    "南京": {"today": 20, "tomorrow": 23},
}

SUPPORTED_CITIES = list(FAKE_WEATHER.keys())


# ----------------------
# Tools
# ----------------------
def calculator(expression: str) -> str:
    allowed = "0123456789+-*/(). "
    if any(ch not in allowed for ch in expression):
        return "Error: invalid characters"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"Error: {e}"


def get_weather(city: str, day: str = "tomorrow") -> str:
    city_data = FAKE_WEATHER.get(city)
    if city_data is None:
        return f"No weather data for {city}"

    temp = city_data.get(day)
    if temp is None:
        return f"No weather data for {city} on {day}"

    return f"{city} {day} temperature is {temp} C"


TOOLS = {
    "calculator": calculator,
    "get_weather": get_weather,
}


# ----------------------
# Extraction helpers
# ----------------------
def extract_cities(user_query: str) -> List[str]:
    found = []
    for city in SUPPORTED_CITIES:
        if city in user_query:
            found.append(city)
    return found


def extract_day(user_query: str) -> str:
    if "今天" in user_query:
        return "today"
    elif "明天" in user_query:
        return "tomorrow"
    return "tomorrow"   # default


def extract_expressions(user_query: str) -> List[str]:
    matches = re.findall(r"[0-9+\-*/(). ]+", user_query)
    matches = [x.strip() for x in matches if x.strip() and any(ch.isdigit() for ch in x)]
    return matches


# ----------------------
# Planner
# ----------------------
def decide_actions(user_query: str) -> List[Dict[str, Any]]:
    actions = []

    # weather-related intent
    if "天气" in user_query or "气温" in user_query:
        cities = extract_cities(user_query)
        day = extract_day(user_query)

        for city in cities:
            actions.append({
                "tool": "get_weather",
                "args": {"city": city, "day": day}
            })

    # calculator-related intent
    expressions = extract_expressions(user_query)
    if expressions:
        expression = max(expressions, key=len)  # still simple heuristic
        actions.append({
            "tool": "calculator",
            "args": {"expression": expression}
        })

    return actions


# ----------------------
# Agent loop
# ----------------------
def custom_agent(user_query: str) -> str:
    short_term_memory = {
        "user_query": user_query,
        "tool_results": []
    }

    actions = decide_actions(user_query)

    for action in actions:
        tool_name = action["tool"]
        args = action["args"]

        result = TOOLS[tool_name](**args)

        short_term_memory["tool_results"].append({
            "tool": tool_name,
            "args": args,
            "result": result
        })

    weather_info = []
    calc_info = []

    for item in short_term_memory["tool_results"]:
        if item["tool"] == "get_weather":
            weather_info.append(item)
        elif item["tool"] == "calculator":
            calc_info.append(item)

    final_parts = []

    # weather outputs
    if weather_info:
        temps = []
        for item in weather_info:
            final_parts.append(item["result"])
            m = re.search(r"is (\d+) C", item["result"])
            if m:
                temps.append((item["args"]["city"], int(m.group(1))))

        if len(temps) >= 2:
            diff = abs(temps[0][1] - temps[1][1])
            final_parts.append(
                f"Temperature difference between {temps[0][0]} and {temps[1][0]} is {diff} C."
            )

    # calculator outputs
    for item in calc_info:
        expr = item["args"]["expression"]
        final_parts.append(f"{expr} = {item['result']}")

    if not final_parts:
        return "I could not determine which tools to use."

    return "\n".join(final_parts)


# test
print(custom_agent("北京和上海明天气温差多少？顺便算一下 25*17"))
print(custom_agent("广州和深圳今天天气怎么样？再算一下 12/3+5")) 

Question: Not general still text-based agent, realying on the query style and the expression formulate 


- Planner is if- else based. should change to 
query → 抽取实体/参数 → 生成 actions → 调用 tools → 汇总结果 

Above code the query is fixed   

第一阶段：规则泛化， but the problem is 
    - 城市还是词典匹配. Not from API 
    - query 说法一换，可能识别不到 By detecting the key words for getting the action planner
    - 没有真正 reasoning 
    - 没有“中间表示层” 
 
第二阶段：planner 输出标准 action schema.
    - Not parser the query to get the key word.  
    - Get the structured representation 结构化语义表示
        - Add the request for extra questions 
        - Add another actions for the request type 
        - 

第三阶段：改成 LLM-based planner . Still not LLM depednent 
 
    - No need to parse the data or use the structured plan to extract the information, formulate into a list of the requests. Here just use the genai  to give the structured plan. 

        - LLm output the structured plan 
        - LLM gives the actions list 

    Directly 

 


## 第二阶段：planner 输出标准 action schema
- Formulate the query text into the dictionary as the plan object
{
    "weather_request": {
        "cities": ["北京", "上海"],
        "day": "tomorrow",
        "compare": True
    } ,
    "calculation_request": {
        "expression": "25*17"
    }
}

- from the plan object 生成 actions：
Here the actions list is the same, the list of the dictionary with the name of the action  and the arguments as the keys. 

[
    {"tool": "get_weather", "args": {"city": "北京", "day": "tomorrow"}},
    {"tool": "get_weather", "args": {"city": "上海", "day": "tomorrow"}},
    {"tool": "calculator", "args": {"expression": "25*17"}}
] 

In [ ]:

def parse_query(user_query: str) -> Dict[str, Any]:
    plan = {
        "weather_request": None,
        "calculation_request": None,
    }

    # weather intent
    if any(word in user_query for word in ["天气", "气温", "多少度", "冷", "热"]):
        cities = extract_cities(user_query)
        if cities:
            plan["weather_request"] = {
                "cities": cities,
                "day": extract_day(user_query),
                "compare": ("差" in user_query or "比较" in user_query or "比" in user_query)
            }

    # calculation intent
    expr = extract_expression(user_query)
    if expr:
        plan["calculation_request"] = {
            "expression": expr
        }

    return plan 



# build the action from the parsed query 
# ----------------------
# Build actions from structured plan
# ----------------------
def build_actions_from_plan(plan: Dict[str, Any]) -> List[Dict[str, Any]]:
    actions = []

    weather_request = plan.get("weather_request")
    if weather_request:
        for city in weather_request["cities"]:
            actions.append({
                "tool": "get_weather",
                "args": {
                    "city": city,
                    "day": weather_request["day"]
                }
            })

    calc_request = plan.get("calculation_request")
    if calc_request:
        actions.append({
            "tool": "calculator",
            "args": {
                "expression": calc_request["expression"]
            }
        })

    return actions




# Run the action to store the reuslt in the observation or short memory 
# list, including the action name and args from the actions , result, .
# Thus {["action": action, "result": result]} 
# ----------------------
# Execute actions
# ----------------------
def run_actions(actions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    observations = []

    for action in actions:
        tool_name = action["tool"]
        args = action["args"]
        result = TOOLS[tool_name](**args)

        observations.append({
            "action": action,
            "result": result
        })

    return observations
 
 
 
 
 # ----------------------
# Final synthesis
# ----------------------
def synthesize_answer(plan: Dict[str, Any], observations: List[Dict[str, Any]]) -> str:
    final_parts = []

    weather_obs = [x for x in observations if x["action"]["tool"] == "get_weather"]
    calc_obs = [x for x in observations if x["action"]["tool"] == "calculator"]

    if weather_obs:
        temps = []
        for obs in weather_obs:
            text = obs["result"]
            final_parts.append(text)

            m = re.search(r"is (\d+) C", text)
            if m:
                temps.append((obs["action"]["args"]["city"], int(m.group(1))))

        weather_request = plan.get("weather_request")
        if weather_request and weather_request["compare"] and len(temps) >= 2:
            diff = abs(temps[0][1] - temps[1][1])
            final_parts.append(
                f"Temperature difference between {temps[0][0]} and {temps[1][0]} is {diff} C."
            )

    for obs in calc_obs:
        expr = obs["action"]["args"]["expression"]
        final_parts.append(f"{expr} = {obs['result']}")

    if not final_parts:
        return "I could not understand the request."

    return "\n".join(final_parts)

# 第三阶段：改成 LLM-based planner  
# It has name, description, parameteris and the required 
 
Exp：比如用户说： 

    北京和上海明天气温差多少？顺便算一下 25*17
    
    1. plan is to output the structured plan。 llm 根据prompt 得出一个plan （根据query 得到的）， python run the plan 调用相应的action。 然后存结果
        - Give the prompt like ： 
        
            You are a planner.

            Available tools:

            1. get_weather(city: string, day: string)
            Use this tool to get weather for a city and day.

            2. calculator(expression: string)
            Use this tool to evaluate a math expression.

            Given the user query, output a JSON array of tool calls.

            User query:
            北京和上海明天气温差多少？顺便算一下 25*17。

        - LLM output
            [
            {"tool": "get_weather", "args": {"city": "北京", "day": "tomorrow"}},
            {"tool": "get_weather", "args": {"city": "上海", "day": "tomorrow"}},
            {"tool": "calculator", "args": {"expression": "25*17"}}
            ]


    2. **Give the tools schema**， with the tools list。 
    方法更好
     TOOLS = {
    "get_weather": get_weather,
    "calculator": calculator,
    }

        - Exp schema input into LLM 
            tools = [
            {
                "name": "get_weather",
                "description": "Get weather for a city on a given day",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string"},
                        "day": {"type": "string"}
                    },
                    "required": ["city", "day"]
                }
            },
            {
                "name": "calculator",
                "description": "Evaluate a math expression",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "expression": {"type": "string"}
                    },
                    "required": ["expression"]
                }
            }
        ]

        - LLM output response.tool_calls

                {
        "tool_calls": [
            {
            "name": "get_weather",
            "arguments": {"city": "北京", "day": "tomorrow"}
            },
            {
            "name": "get_weather",
            "arguments": {"city": "上海", "day": "tomorrow"}
            },
            {
            "name": "calculator",
            "arguments": {"expression": "25*17"}
            }
        ]
        }
                    

In [ ]:

TOOL_SCHEMAS = [
    {
        "name": "get_weather",
        "description": "Get weather for a city on a given day",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "day": {"type": "string"}
            },
            "required": ["city", "day"]
        }
    },
    {
        "name": "calculator",
        "description": "Evaluate a math expression",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string"}
            },
            "required": ["expression"]
        }
    }
    

]



# ----------------------
# Simulated LLM planner
# In real use, this would be an API call
# Here the llm plan is fixed use the if-else to get  the cities and the expression
# ----------------------
def llm_plan(user_query: str, tool_schemas: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    这里先假装 LLM 已经输出结构化 action plan。
    真实系统里，这里会调用 OpenAI / Gemini / local model。
    """
    # 假设模型输出 for the prompt 
    if "北京" in user_query and "上海" in user_query and "25*17" in user_query:
        return [
            {"tool": "get_weather", "args": {"city": "北京", "day": "tomorrow"}},
            {"tool": "get_weather", "args": {"city": "上海", "day": "tomorrow"}},
            {"tool": "calculator", "args": {"expression": "25*17"}},
        ]
    # return the json form for the subsequent parse to call the function 
    return [] 


# ----------------------
# Execute
# ----------------------
def run_actions(actions: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    observations = []
    for action in actions:
        tool_name = action["tool"]
        args = action["args"]
        result = TOOLS[tool_name](**args)
        observations.append({
            "action": action,
            "result": result
        })
    return observations





def custom_agent_v3(user_query: str) -> str:
    actions = llm_plan(user_query, TOOL_SCHEMAS)
    observations = run_actions(actions)
    answer = llm_summarize(user_query, observations)
    return answer 



# ----------------------
# Simulated final LLM answer
# ----------------------
def llm_summarize(user_query: str, observations: List[Dict[str, Any]]) -> str:
    lines = []
    for obs in observations:
        lines.append(obs["result"])
    return "\n".join(lines)
 
 


## How to do memory 

- API。 the model takes the message as input
    - Short memory 
        - exp
        messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "My name is Alice."},
        {"role": "assistant", "content": "Nice to meet you, Alice."},
        {"role": "user", "content": "What is my name?"}
        ]

        - tool calling memory： 
            - add the previous_response_id=response1.id into the openai chat model create 
            
            - call the llm again with tool_outputs 作为 observation 回传给模型 

    - Long term memory 

        - input part is current query information
        - part is the longterm memory， the longterm memory is inputed from the longmemory file 
        
- 本地模型：通常没有“自动记忆”，你要自己维护 messages，再用 apply_chat_template 把它们格式化成模型输入 。 LLM only do the output generation in text。 

    - exp with the messages to be tokenizered and generate the output ， then  messages.append({"role": "assistant", "content": answer}) 

    - tool calling message
        - Add the text into the model



 

In [ ]:
# 
import json
from pathlib import Path

MEMORY_FILE = Path("memory.json")

def load_memory():
    if MEMORY_FILE.exists():
        return json.loads(MEMORY_FILE.read_text())
    return {}

def save_memory(memory):
    MEMORY_FILE.write_text(json.dumps(memory, ensure_ascii=False, indent=2))

def remember_fact(key: str, value: str):
    memory = load_memory()
    memory[key] = value
    save_memory(memory)

def recall_fact(key: str):
    memory = load_memory()
    return memory.get(key)

remember_fact("favorite_food", "sushi")
print(recall_fact("favorite_food"))